# Fig. 2f — Dot plot of mechanically and voltage-gated ion channels across VHC subtypes

**Input**
- `adata_all_final.h5ad` — merged VGN + VHC AnnData. We subset to `batch == 'HC_ss3'` (VHC only).
- `selected_VHCs.xlsx` — curated marker panel; row index = gene symbol. Contents are exclusively mechanically + voltage-gated ion channels (Cacn\*, Kcn\*, Hcn\*), MET machinery (Tmc1/2, Tmie, Lhfpl5, Loxhd1), MET-anchored complex (Cib2/3, Myo7a, Myo15a/b), and the IHC-style synaptic glutamate transporter Slc17a8.

**Output**
- `figures/Fig2f_VHC_ion_channels_dotplot.pdf`

**Notes**
- Dot size = fraction of cells expressing (`expression_cutoff=0`); dot colour = within-gene z-scored mean expression (`standard_scale='var'`).
- `min_frac=0.23` removes genes that are not detectable above 23% in any single VHC subtype — these are uninformative for distinguishing subtypes.
- Hand-curated removal list (`remove_list`) drops a few channels that look broadly expressed without subtype specificity (`Kcnc2/3`, `Kcnd3`, `Cachd1`, `Cacnb4`).
- Both rows (genes) and columns (clusters) are hierarchically clustered (Euclidean, average linkage) — the resulting dendrograms reflect functional grouping of channels and similarity between VHC subtypes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm

from _dotplot import marker_dotmap_simple

sc.settings.verbosity = 1
sc.set_figure_params(figsize=(5, 5), dpi_save=300, dpi=100, frameon=False)
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.family'] = 'Arial'

DATA_DIR = Path('..')
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

cmap_wdg = mcolors.LinearSegmentedColormap.from_list(
    'white_darkgrey', cm.Greys(np.linspace(0.0, 0.70, 256))
)

## 1. Load VHC subset and the pre-selected marker panel

In [ ]:
adata_all = sc.read_h5ad(DATA_DIR / 'adata_all_final.h5ad')
adata_all.X = adata_all.layers['umi'].copy()  # dot plot uses raw counts via use_raw=True downstream

adata_HC_ss3 = adata_all[adata_all.obs['batch'].isin(['HC_ss3'])].copy()
print(adata_HC_ss3)
print('\nVHC subtypes (cells):')
print(adata_HC_ss3.obs['all_cluster_gene_name'].value_counts())

In [ ]:
slct_genes = pd.read_excel(DATA_DIR / 'selected_VHCs.xlsx', index_col=0).index.tolist()

# Hand-curated removals: broadly expressed channels without subtype specificity
remove_list = ['Kcnc2', 'Kcnc3', 'Kcnd3', 'Cachd1', 'Cacnb4']
slct_genes = [g for g in slct_genes if g not in remove_list]

print(f'{len(slct_genes)} genes in panel after curation')
print(slct_genes)

## 2. Plot

In [ ]:
fig = marker_dotmap_simple(
    adata_HC_ss3, slct_genes,
    groupby='all_cluster_gene_name',
    color_map=cmap_wdg,
    use_raw=True, standard_scale='var',
    expression_cutoff=0.0, min_frac=0.23,
    fontsize=12, spines=False,
    rows_cluster=True, cols_cluster=True,
    cluster_method='average', cluster_metric='euclidean',
)
plt.savefig(FIG_DIR / 'Fig2f_VHC_ion_channels_dotplot.pdf', dpi=600, bbox_inches='tight')
plt.show()